In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [2]:
csv_path = "stg_appointments.csv"

df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(110521, 22)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,...,SMS_received,NoShowFlag,WaitingDays,WaitingGroup,ObservedVisitType,IncomeMean,IncomeMedian,ResponsiblePersons,Residents,IncomeGroup
0,39217.84439,5751990,F,2016-05-31 10:56:41+00:00,2016-06-03 00:00:00+00:00,44,PRAIA DO SUÁ,0,0,0,...,0,0,3,3-7 days,First observed appointment,4991.21,2500.0,1053.0,2720.0,Upper-middle income
1,43741.75652,5760144,M,2016-06-01 14:22:58+00:00,2016-06-01 00:00:00+00:00,39,MARIA ORTIZ,0,0,1,...,0,0,0,Same day,First observed appointment,2377.33,1800.0,4779.0,12534.0,Upper-middle income
2,93779.52927,5712759,F,2016-05-18 09:12:29+00:00,2016-05-18 00:00:00+00:00,33,CENTRO,0,0,0,...,0,0,0,Same day,First observed appointment,4390.34,3001.0,3991.0,9100.0,Higher income
3,141724.16655,5637648,M,2016-04-29 07:13:36+00:00,2016-05-02 00:00:00+00:00,12,FORTE SÃO JOÃO,0,0,0,...,0,0,3,3-7 days,First observed appointment,1942.61,1300.0,768.0,2056.0,Lower income
4,537615.28476,5637728,F,2016-04-29 07:19:57+00:00,2016-05-06 00:00:00+00:00,14,FORTE SÃO JOÃO,0,0,0,...,1,0,7,3-7 days,First observed appointment,1942.61,1300.0,768.0,2056.0,Lower income


In [22]:
import os
from dotenv import load_dotenv

load_dotenv()

server = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")
username = os.getenv("SQL_USERNAME")
password = os.getenv("SQL_PASSWORD")

In [6]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server']


In [9]:
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=60;"
)

connection_url = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_url}",
    fast_executemany=True
)

In [10]:
with engine.connect() as conn:
    result = conn.exec_driver_sql(
        "SELECT DB_NAME() AS CurrentDatabase"
    )

    print(result.fetchone())

('sql-healthcare',)


In [11]:
df = pd.read_csv("stg_appointments.csv")

print(df.shape)
print(df.columns.tolist())

(110521, 22)
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'NoShowFlag', 'WaitingDays', 'WaitingGroup', 'ObservedVisitType', 'IncomeMean', 'IncomeMedian', 'ResponsiblePersons', 'Residents', 'IncomeGroup']


In [12]:
query = """
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    ORDINAL_POSITION
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'stg_Appointments'
ORDER BY ORDINAL_POSITION;
"""

sql_columns = pd.read_sql(query, engine)

sql_columns

,COLUMN_NAME,DATA_TYPE,ORDINAL_POSITION
0,PatientId,varchar,1
1,AppointmentID,bigint,2
2,Gender,char,3
3,ScheduledDay,datetime2,4
4,AppointmentDay,date,5
5,Age,int,6
6,Neighbourhood,varchar,7
7,Scholarship,bit,8
8,Hypertension,bit,9
9,Diabetes,bit,10


In [13]:
print("CSV columns:")
print(df.columns.tolist())

print("\nSQL columns:")
print(sql_columns['COLUMN_NAME'].tolist())

CSV columns:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'NoShowFlag', 'WaitingDays', 'WaitingGroup', 'ObservedVisitType', 'IncomeMean', 'IncomeMedian', 'ResponsiblePersons', 'Residents', 'IncomeGroup']

SQL columns:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received', 'NoShowFlag', 'WaitingDays', 'WaitingGroup', 'ObservedVisitType', 'IncomeMean', 'IncomeMedian', 'ResponsiblePersons', 'Residents', 'IncomeGroup']


In [14]:
df.to_sql(
    name="stg_Appointments",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=1000
)

DatabaseError: Execution failed on sql 'INSERT INTO dbo."stg_Appointments" ("PatientId", "AppointmentID", "Gender", "ScheduledDay", "AppointmentDay", "Age", "Neighbourhood", "Scholarship", "Hypertension", "Diabetes", "Alcoholism", "Handicap", "SMS_received", "NoShowFlag", "WaitingDays", "WaitingGroup", "ObservedVisitType", "IncomeMean", "IncomeMedian", "ResponsiblePersons", "Residents", "IncomeGroup") VALUES (:PatientId, :AppointmentID, :Gender, :ScheduledDay, :AppointmentDay, :Age, :Neighbourhood, :Scholarship, :Hypertension, :Diabetes, :Alcoholism, :Handicap, :SMS_received, :NoShowFlag, :WaitingDays, :WaitingGroup, :ObservedVisitType, :IncomeMean, :IncomeMedian, :ResponsiblePersons, :Residents, :IncomeGroup)': (pyodbc.ProgrammingError) ('String data, right truncation: length 50 buffer 20', 'HY000')
[SQL: INSERT INTO dbo.[stg_Appointments] ([PatientId], [AppointmentID], [Gender], [ScheduledDay], [AppointmentDay], [Age], [Neighbourhood], [Scholarship], [Hypertension], [Diabetes], [Alco

In [15]:
df = pd.read_csv(
    "stg_appointments.csv",
    dtype={
        "PatientId": "string"
    }
)

In [16]:
df['ScheduledDay'] = (
    pd.to_datetime(df['ScheduledDay'], utc=True)
    .dt.tz_localize(None)
)

df['AppointmentDay'] = (
    pd.to_datetime(df['AppointmentDay'], utc=True)
    .dt.tz_localize(None)
)

In [17]:
print(df['PatientId'].dtype)
print(df['ScheduledDay'].dtype)
print(df['AppointmentDay'].dtype)

df[
    ['PatientId', 'ScheduledDay', 'AppointmentDay']
].head()

string
datetime64[us]
datetime64[us]


,PatientId,ScheduledDay,AppointmentDay
0,39217.84439,2016-05-31 10:56:41,2016-06-03
1,43741.75652,2016-06-01 14:22:58,2016-06-01
2,93779.52927,2016-05-18 09:12:29,2016-05-18
3,141724.16655,2016-04-29 07:13:36,2016-05-02
4,537615.28476,2016-04-29 07:19:57,2016-05-06


In [18]:
print(df.shape)

print(df[
    ['PatientId', 'ScheduledDay', 'AppointmentDay']
].isna().sum())

(110521, 22)
PatientId         0
ScheduledDay      0
AppointmentDay    0
dtype: int64


In [19]:
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_url}"
)

In [20]:
df.to_sql(
    name="stg_Appointments",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=500
)

5546

In [21]:
with engine.connect() as conn:
    result = conn.exec_driver_sql(
        "SELECT COUNT(*) FROM dbo.stg_Appointments"
    )
    print(result.fetchone())

(110521,)


In [23]:
print(server)
print(database)
print(username)
print(password is not None)

sql-healthcare-noshow.database.windows.net
sql-healthcare
leao
True
